# Рабочие профессии: пять поисковых экспериментов

Русскоязычный профиль содержит роли, оборудование, графики, условия и исправления ввода.
C1 — текст, C2 — ограниченные сигналы, C3 — приоритет графика, C4 — понятия, C5 — исправления.
После изменения профиля полей или коэффициентов повторите сборку. Просмотр DSL не делает HTTP.

Подключение OpenSearch описано в [руководстве](../../../docs/usage.md#поиск-рабочих-профессий). Fixture-тесты проверяют передачу запросов, но не исполняют Painless. Авторизация берётся из `OPENSEARCH_AUTHORIZATION` окружения kernel.


In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/main.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from src.components.reporting.exporters.csv import SearchCSVExporter
from src.components.reporting.exporters.json import JSONExporter
from src.experiments.batch.search import BatchSearchExperiment

## Параметры

Имена полей — пример. Укажите существующие single-valued поля с doc values.
Без метаданных выберите `variants = ["c1", "c4", "c5"]` и удалите override `ranking_fields`. Для C2 доступен `freshness-only` без quality.

После изменений повторите **compose**, затем instantiate: Hydra может материализовать внешние интерполяции, поэтому правки уже разрешённого cfg не пересобирают связи.


In [ ]:
source_overrides = [
    "connection.url=http://localhost:9200",
    "jobboard.source.index=vacancies",
    "jobboard.source.fields=[title,description]",
    "jobboard.source.title_field=title",
    "+jobboard.source.ranking_fields={published_at:{field:published_at,type:date},quality:{field:card_completeness,type:numeric},category:{field:schedule_category,type:keyword},stable_id:{field:vacancy_id,type:keyword}}",
]
variants = ["c1", "c2", "c3", "c4", "c5"]
now_ms = int(time.time() * 1000)  # фиксируется на всю сравнительную сессию
text = "комплектовшик ТСД без опыта"
experiment_overrides = []
# Например: ["jobboard/lexical/research/ablation@_global_=no-aliases"]
# Или: ["jobboard/lexical/ranking@jobboard.rankings.signals=freshness-only"]
input_path = ROOT / "examples/blue-collar/queries.csv"
output_dir = ROOT / "outputs" / f"work-research-{now_ms}"

In [ ]:
with initialize_config_dir(config_dir=str(ROOT / "configs"), version_base="1.3"):
    cfg = compose(
        config_name="main",
        overrides=[
            "~experiments/vacancy-search/interactive@experiment",
            "+experiments/vacancy-search/jobboard/research@experiment=001-compare",
            *source_overrides,
            f"jobboard.session.now_ms={now_ms}",
            "experiment.variants=[" + ",".join(variants) + "]",
            "experiment.exporters=[]",
            *experiment_overrides,
        ],
    )
    experiment = instantiate(cfg.experiment, _convert_="all")
print("Собраны варианты:", experiment.variants)

## Разбор запроса и DSL

Hints содержат span и rule_id, rewrite_candidates — принятые и отклонённые замены.
`unsupported_salary_constraint` означает, что фильтр диапазона зарплаты не поддерживается. Исходный запрос сохраняется. Ячейка работает без сети.


In [ ]:
plans = experiment.preview(text)
for variant, plan in plans.items():
    print(variant)
    print(
        json.dumps(
            {"understanding": plan.understanding, "request_body": plan.body},
            ensure_ascii=False,
            indent=2,
        )
    )

## Один запрос во всех вариантах

Ячейка выполняет HTTP. Значение score зависит от режима: C2 возвращает script score, C3 — lexical score с отдельным sort key, rescore — суммарную оценку.
F/Q можно проверить по полям и params. Отдельный вклад phrase score в ответе отсутствует.


In [ ]:
artifact = experiment.compare(text)
artifact

In [ ]:
artifact.rank_changes()

## Подборки запросов

Сравнивайте профессии и условия в выдаче. `query_sets` содержит короткие подборки, полный CSV — более 200 локально проверенных входов. Успешный parse не подтверждает качество удалённого поиска.
Неизвестное поле или запрет scripts завершает запрос ошибкой; проверьте конфигурацию и возможности индекса.


In [ ]:
query_sets = {
    "склад": ["грузчик", "водитель погрузчика", "комплектовщик ТСД", "кладовщик WMS"],
    "водители": [
        "водитель категории B",
        "водитель категории CE",
        "пеший курьер",
        "курьер на своём автомобиле",
    ],
    "производство": ["оператор ЧПУ", "наладчик ЧПУ", "сварщик НАКС", "токарь 5 разряд"],
    "торговля": [
        "кассир",
        "продавец кассир",
        "работник торгового зала",
        "мерчандайзер",
    ],
    "условия": [
        "грузчик ночные смены",
        "бетонщик вахта 30/30",
        "охранник без лицензии",
        "водитель от 100000 на руки",
    ],
    "ошибки ввода": [
        "вадитель категории C",
        "rkfljdobr Москва",
        "svarshik вахта",
        "e,jhobr офиса",
    ],
}
query_sets

## Batch и сохранение

Матрица выполняет `число запросов × число вариантов` поисков и хранит результаты в памяти. Экспорт начинается после полного успеха и сохраняет существующий файл.
Артефакт содержит тексты запросов и документы, без credentials и connection config.


In [ ]:
from src.experiments.batch.search import read_queries

queries = read_queries(input_path)
batch_artifact = experiment.compare_queries(queries)
output_dir.mkdir(parents=True, exist_ok=True)
JSONExporter(str(output_dir / "research.json")).export(batch_artifact)
print("Сохранено:", output_dir / "research.json")

## CSV одного варианта

Формат [spec 005](../../../docs/specs/005-search-research.md) сохраняет выдачи без sort/parse diagnostics. Полная диагностика доступна в research.json выше.
Выбранный вариант исполняется тем же SearchExperiment.


In [ ]:
single = BatchSearchExperiment(experiment.searches[variants[0]], str(input_path))
SearchCSVExporter(str(output_dir / "searches.csv")).export(single.run())